# Zaskaleta AI Twin — AUTO v4

## ▶ ЗАПУСК З ТЕЛЕФОНУ
**Натисни меню `Виконання / Runtime` → `Виконати все / Run all`.**

AUTO v4: вибір ОСНОВНОГО Google-акаунта → checkpoint/resume → voice → scenes → MuseTalk → final 9:16 MP4 → архів на другий Google Drive.

На старті AUTO відкриває окрему авторизацію Google. Вибери акаунт, де лежать 6 master photos, master voice, behavior videos і Day_01. Якщо вибрано не той акаунт, AUTO дозволить повторити вибір без інсталяції.

Другий Google Drive авторизується тільки ПІСЛЯ готового фінального відео.

**T4 GPU має бути увімкнений.**


In [ ]:
import os, re, subprocess, sys, json
from pathlib import Path
import torch

GPU_OK=torch.cuda.is_available()
print('CUDA:',GPU_OK)
if not GPU_OK:
    print('\n⛔ GPU ЗАРАЗ НЕДОСТУПНИЙ У COLAB')
    raise SystemExit(0)
print('✅ T4/CUDA доступний — продовжуємо AUTO v4')

from google.colab import auth, drive

def has_source_assets(root=Path('/content/drive/MyDrive')):
    pats=['Zaskaleta_AI_Voice_Master.mp3','Zaskaleta_AI_Voice_Master.wav','Zaskaleta_AI_Voice_Master.m4a','Zaskaleta_AI_Voice_Master.flac','Zaskaleta_AI_Voice_Master.aac','Zaskaleta_AI_Voice_Master.ogg']
    for name in pats:
        try:
            if any(root.rglob(name)):
                return True
        except Exception:
            pass
    return False

selected=False
for attempt in range(1,4):
    print(f'\n🔐 ВИБІР ОСНОВНОГО GOOGLE-АКАУНТА — спроба {attempt}/3')
    print('У вікні Google вибери акаунт, де є 6 master photos, master voice, behavior videos і Day_01.')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        auth.authenticate_user(clear_output=False)
    except TypeError:
        auth.authenticate_user()
    drive.mount('/content/drive', force_remount=True)
    if has_source_assets():
        print('✅ ПРАВИЛЬНИЙ ОСНОВНИЙ GOOGLE DRIVE — source assets знайдені')
        selected=True
        break
    print('❌ На цьому акаунті master voice не знайдено.')
    if attempt < 3:
        input('Натисни Enter, щоб ВИБРАТИ ІНШИЙ GOOGLE-АКАУНТ...')

if not selected:
    print('⛔ Після 3 спроб правильний основний Drive не вибрано. Інсталяцію не запускаємо.')
    raise SystemExit(0)

ARCHIVE_CFG=Path('/content/drive/MyDrive/Zaskaleta_AI_Twin_archive_config.json')
DEFAULT_ARCHIVE_FOLDER='https://drive.google.com/drive/folders/1_7G-rAGQ80Vpe_CWdGOzPIg0nuprDp3s'
archive_folder=DEFAULT_ARCHIVE_FOLDER
if ARCHIVE_CFG.is_file():
    try:
        saved=json.loads(ARCHIVE_CFG.read_text(encoding='utf-8')).get('folder','').strip()
        if saved:
            archive_folder=saved
    except Exception:
        pass
else:
    ARCHIVE_CFG.write_text(json.dumps({'folder':archive_folder},ensure_ascii=False,indent=2),encoding='utf-8')
print('💾 Архівний Drive налаштовано; авторизацію перенесено на кінець рендера')

ROOT=Path('/content/zaskaleta-ai-twin-colab')
if ROOT.exists():
    subprocess.run(['rm','-rf',str(ROOT)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git',str(ROOT)],check=True)

WORKER=ROOT/'worker'
env=os.environ.copy()
env['APP_DIR']=str(WORKER)
env['MUSETALK_ROOT']='/content/MuseTalk'
env['VENV_DIR']='/content/ai-twin-py311'

print('\n========== INSTALLER START ==========')
iproc=subprocess.Popen(['bash',str(WORKER/'install_gpu_engines.sh')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
install_lines=[]
for line in iproc.stdout:
    print(line,end='')
    install_lines.append(line)
icode=iproc.wait()
if icode != 0:
    tail=''.join(install_lines[-80:])
    print('\n❌ INSTALLER FAILED — last output:\n'+tail)
    raise RuntimeError(f'Installer stopped with exit code {icode}')
print('========== INSTALLER OK ==========\n')

PY='/content/ai-twin-py311/bin/python'
cmd=[PY,str(WORKER/'run_auto_v3.py'),'--root',str(ROOT),'--mydrive','/content/drive/MyDrive']
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
for line in proc.stdout:
    print(line,end='')
    lines.append(line)
code=proc.wait()
if code != 0:
    print('\n❌ AUTO PIPELINE FAILED — last output:\n'+''.join(lines[-80:]))
    raise RuntimeError(f'AUTO v4 stopped with exit code {code}')
out=''.join(lines)
m=re.search(r'^FINAL_PATH=(.+)$',out,re.MULTILINE)
if not m:
    raise RuntimeError('AUTO v4 завершився без FINAL_PATH')
FINAL=m.group(1).strip()
final_path=Path(FINAL)
print('✅ FINAL READY:',FINAL)

if archive_folder:
    try:
        print('\n🔐 Фінальне відео готове. Тепер вибери ДРУГИЙ Google-акаунт для архіву.')
        try:
            auth.authenticate_user(clear_output=False)
        except TypeError:
            auth.authenticate_user()
        day_match=re.search(r'Day_(\d{2})',final_path.name) or re.search(r'Day_(\d{2})',str(final_path.parent))
        archive_day=int(day_match.group(1)) if day_match else 0
        subprocess.run([sys.executable,str(WORKER/'archive_second_drive.py'),'--folder',archive_folder,'--episode-dir',str(final_path.parent),'--final',str(final_path),'--day',str(archive_day)],check=True)
        print('✅ Готове відео та метадані збережено на другому Google Drive')
    except Exception as e:
        print('⚠️ Архівація на другий Drive не вдалася.')
        print('✅ Фінальний файл на основному Drive збережений і не пошкоджений.')
        print('Причина:',e)

from IPython.display import Video, display
print('✅ FINAL:',FINAL)
display(Video(FINAL,embed=True,width=360))
